# Snowflake Cloud Data Warehousing Fundamentals

Welcome to **Project 12: Snowflake for Data Engineering** in the *Data Engineering Diaries* series by Sathiya Murthy.

This interactive notebook demonstrates key Snowflake capabilities:
1. **Architecture & Connection Management**
2. **Data Staging & Bulk Ingestion (`COPY INTO`)**
3. **Change Data Capture (CDC) with Streams & Tasks**
4. **Continuous Pipelines with Dynamic Tables**
5. **Time Travel & Zero-Copy Cloning**
6. **Extensibility with Snowpark Python UDFs**
7. **FinOps & Cost Optimization**

## 1. Initializing Connection & Environment Config

In [ ]:
import sys
import os
sys.path.append('..')

from src.snowflake_connector import SnowflakeConnector
from src.snowflake_utils import SnowflakeUtils
from src.data_loader import SnowflakeDataLoader
from src.snowflake_etl import SnowflakeETLPipeline
from src.cost_monitor import SnowflakeCostMonitor

# Initialize connector with fallback mock support
connector = SnowflakeConnector(force_mock=True)
config = connector.get_connection_config()
print("Active Connection Config:", config)
print("Is Connection Mocked?:", connector.is_mock())

## 2. Data Staging & Bulk Loading Pipeline

In [ ]:
loader = SnowflakeDataLoader(connector=connector)

sample_sales_path = "../sample-data/sales_data.csv"
if os.path.exists(sample_sales_path):
    load_result = loader.load_csv_pipeline(
        local_csv_path=sample_sales_path,
        target_table="STG_SALES",
        stage_name="RAW_INTERNAL_STAGE",
        format_name="CSV_HEADER_FORMAT"
    )
    print("Pipeline Load Summary:", load_result)
else:
    print(f"Sample data file not found at {sample_sales_path}")

## 3. Change Data Capture with Streams and Tasks

In [ ]:
etl = SnowflakeETLPipeline(connector=connector)

# Create Stream on Staging Table
stream_res = etl.create_stream(stream_name="STG_SALES_STREAM", source_table="STG_SALES", stream_type="STANDARD")
print("Stream Created:", stream_res)

# Generate MERGE SQL statement
merge_sql = etl.build_merge_statement(
    target_table="ANALYTICS.FACT_SALES",
    stream_name="STG_SALES_STREAM",
    join_keys=["transaction_id"],
    update_columns=["quantity", "total_amount"],
    insert_columns=["transaction_id", "customer_id", "product_id", "category", "quantity", "unit_price", "total_amount", "payment_method", "store_id", "transaction_timestamp"]
)
print("Generated CDC Merge SQL:\n", merge_sql)

## 4. Querying Metadata & Warehouse Inspection

In [ ]:
utils = SnowflakeUtils(connector=connector)

# Execute multi-statement SQL script
script = """
CREATE TABLE IF NOT EXISTS DEMO_SUMMARY (id INT, total_sales FLOAT);
INSERT INTO DEMO_SUMMARY VALUES (1, 1299.99), (2, 450.00);
SELECT * FROM DEMO_SUMMARY;
"""
results = utils.execute_script(script)
for res in results:
    print(f"Statement {res['index']}: {res['status']}")

## 5. Cost Monitoring & Right-Sizing Analysis

In [ ]:
monitor = SnowflakeCostMonitor(connector=connector)

# Create Resource Monitor Guardrail
mon_res = monitor.create_resource_monitor(
    monitor_name="DE_DAILY_CAP",
    credit_quota=50,
    frequency="DAILY",
    notify_triggers=[50, 75, 90],
    suspend_trigger=100
)
print("Resource Monitor Status:", mon_res)

# Warehouse Right-Sizing Recommendations
suggestions = monitor.analyze_warehouse_right_sizing()
print("Warehouse Optimization Suggestions:", suggestions)